In [ ]:
# Import all the libraries used in this notebook

import pandas as pd
import matplotlib as plt
from glob import glob
from os.path import join
import numpy as np
from pvlib.solarposition import get_solarposition
import datetime
import os 

import pandas as pd
import plotly.graph_objs as go
import plotly.express as px
import plotly.io as pio

import seaborn as sns
# import matplotlib.pyplot as plt




print(os.environ['CONDA_DEFAULT_ENV']) # Check the name of the current Conda environment

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
maps = pd.read_csv("../../../data/MAPS_20min_CASPER17_West.csv")
rvsr = pd.read_csv("../../../data/RVSR_2017_flux_10_v3_decorr.csv")
flip = pd.read_csv("../../../data/Flip_Flux_Processed_data.csv")

In [ ]:
def drop_variable(df, *args): # delete a variable from the dataframe
    for arg in args:
        df = df.drop(arg, axis=1)

    return df

def dropna_in_variable(df, *args): # delete the rows where a nan exists in the specified column(s)
    df = df.dropna(subset=(args))
    return df

def nan_column_percentage(df, tinterval):
    # Set pandas to not cut out the middle of df in the display results
    pd.set_option('display.max_columns', None)  # Show all columns
    pd.set_option('display.max_rows', None)     # Show all rows (you can set it to a specific number if you want)

    print("Total datapoints/timestamps ({1} min intervals): {0} \n".format(len(df), str(tinterval)))
    print("Column Name                      Percentage of Missing data\n")
    print(df.isna().sum()/len(df)) # shows the percentage of data missing in the corresponding column/variable

## initial analysis

In [ ]:
print(len(maps))
print(len(rvsr))
print(len(flip))

data from rvsr 

is only from 9/30 to 10/26, a little less than a month, timestamps are about 10 mins apart (cycles between 9 min and 11 min, for the ML we can just say 10 and 10)

there are aprox 27 days worth of data, very small for an ML model

data from maps 

is in a weird time stamp
ex:
32:06.1
33:06.1
34:06.1
35:06.1
36:06.1

data points are every minite (mm:ss.milisecond), there is not more time info, so hard to tell when it is a new day
    might to check the documentation of the maps to see if it is specified the start date and time, then we should be able to calculate the rest ourselves.

there are aprox 17 days worth of this data, very small for an ML modelx

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Read the CSV file
maps = pd.read_csv("../../../data/MAPS_20min_CASPER17_West.csv")

# Strip any leading or trailing whitespace from column names
maps.columns = maps.columns.str.strip()

# Ensure 'Time' is a datetime type
maps['Time'] = pd.to_datetime(maps['Time'].str.strip(), errors='coerce')

# Check for rows where 'Time' could not be converted
missing_times = maps[maps['Time'].isna()]
if not missing_times.empty:
    print("Rows with invalid 'Time' values:")
    print(missing_times)

# Drop rows with invalid 'Time' values
maps = maps.dropna(subset=['Time'])

# Set 'Time' as the index
maps.set_index('Time', inplace=True)

# Plot the time series with smaller markers and a thinner line
plt.figure(figsize=(12, 6))
plt.plot(maps.index, maps['air_static_pressure:4.61_m:hPa'], marker='o', linestyle='-', color='b', markersize=0.5, linewidth=0.1)
plt.title('Air Static Pressure Over Time')
plt.xlabel('Time')
plt.ylabel('Air Static Pressure (hPa)')
plt.grid(True)
plt.show()

# Calculate the differences between consecutive timestamps
maps['Time_Diff'] = maps.index.to_series().diff().dt.total_seconds().div(3600)  # difference in hours

# Identify the gaps
gaps = maps[maps['Time_Diff'] > (1/60)]  # gaps greater than 1 minute

# Plot the size of the gaps
plt.figure(figsize=(12, 6))
plt.plot(gaps.index, gaps['Time_Diff'], marker='o', linestyle='None', color='r', markersize=5)
plt.title('Gaps in Time Series')
plt.xlabel('Time')
plt.ylabel('Gap Size (hours)')
plt.grid(True)
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Read the CSV file
maps = pd.read_csv("../../../data/MAPS_20min_CASPER17_West.csv")

# Strip any leading or trailing whitespace from column names
maps.columns = maps.columns.str.strip()

# Ensure 'Time' is a datetime type
maps['Time'] = pd.to_datetime(maps['Time'].str.strip(), errors='coerce')

# Check for rows where 'Time' could not be converted
missing_times = maps[maps['Time'].isna()]
if not missing_times.empty:
    print("Rows with invalid 'Time' values:")
    print(missing_times)

# Drop rows with invalid 'Time' values
maps = maps.dropna(subset=['Time'])

# Set 'Time' as the index
maps.set_index('Time', inplace=True)

# Plot the time series with smaller markers and a thinner line
num = 140
plt.figure(figsize=(12, 6))
plt.plot(maps.index[:num], maps['wave_height:0_m:m'][:num], marker='o', linestyle='-', color='b', markersize=0.5, linewidth=0.1)
plt.title('wave_height:0_m:m')
plt.xlabel('Time')
plt.ylabel('wave_height:0_m:m')
plt.grid(True)
plt.show()

# Calculate the differences between consecutive timestamps
# maps['Time_Diff'] = maps.index.to_series().diff().dt.total_seconds().div(3600)  # difference in hours

# # Identify the gaps
# gaps = maps[maps['Time_Diff'] > (1/60)]  # gaps greater than 1 minute

# # Plot the size of the gaps
# plt.figure(figsize=(12, 6))
# plt.plot(gaps.index, gaps['Time_Diff'], marker='o', linestyle='None', color='r', markersize=5)
# plt.title('Gaps in Time Series')
# plt.xlabel('Time')
# plt.ylabel('Gap Size (hours)')
# plt.grid(True)
# plt.show()


In [ ]:
nan_column_percentage(maps, "supposed to be 20, csv shows 1")

In [ ]:
nan_column_percentage(rvsr, 10)

In [ ]:
nan_column_percentage(flip, 10)

mvco vs rvsr 
zenith 0 vs 0
azimuth 0 vs 0
water surface 0 vs -3.5
pressure 18.4 vs 0 
relative humidity 18.4 vs 12.54
wind speed 18.4 vs 13.49
wind dir 18.4 vs 13.49
air temp (is just called temp) 18.4 vs 12.54

no 
mixing ratio 18.4
    needs at 18.4
    air temp (no)
    relative humidity (yes)
    pressure (yes)
wave dir 0 (no)
wave height 0 (no)
wave period 0 (no)
angle between wind wave 0
    needs
    u and v wave ()
    u and v wind ()
bulk richarson 18.4
    needs 
    potential temp
        needs
        air temp
        pressure (yes)
    mixing ratio
    skin virtual potential temp
        needs 
        water surface temp (yes)
        mixing ratio
    wind speed (yes)

MAPS :          09-28 to 10-23 : 1 min interval
(rvsr file from sue)
FLIPS :         09-30 to 10-23 : seems to be in 15 min intervals, measurements at every level (should be 7 levels, but the .nc file shows a T15)
Radiosonde :    09-26 to 10-25 : 5 measurements per day, aprox every 4.8 hours
RVSR :          09-30 to 10-26 : 10 min interval
(rvsr2 file from sue)

## Graphing with Plotly

In [ ]:
def create_trace_dict(df):
    # Initialize a dictionary to store the traces
    traces = {}

    # Get a list of unique colors
    colors = px.colors.qualitative.Plotly

    # List of strings to exclude
    exclude_strings = ['Time', 'year', 'month', 'day', 'hour', 'minutes', 'seconds']  # replace with your list of strings

    # Loop through each column in the DataFrame, excluding the index column
    for idx, column in enumerate(df.columns):
        if column not in exclude_strings:
            trace = go.Scatter(
                x = df.index,
                y = df[column],
                name = column,
                line = dict(color = colors[idx % len(colors)]),
                connectgaps = False,
                opacity = 0.5
            )
            traces[column] = trace

    # Now you have a dictionary of traces where the key is the column name
    print(traces.keys())
    return traces


In [ ]:
maps_t = create_trace_dict(maps)
rvsr_t = create_trace_dict(rvsr)
flip_t = create_trace_dict(flip)

In [ ]:
import plotly.graph_objs as go
import plotly.io as pio

def draw_traces(traces, num_traces_per_figure = 2, traces_count = 0, figure_count = 1):
    # Assume you have a dictionary 'traces' containing all your traces

    # Initialize dictionary to store traces for each figure
    current_traces = {}  

    # saves image with a name 
    config = {
    'toImageButtonOptions': {
        'format': 'png',  # Default format for the image download button
        'filename': 'figure',  # Default filename
        'height': 720,
        'width': 1280, 
        'scale': 3  # Multiply title/legend/axis/canvas sizes by this factor
    },
    'modeBarButtonsToAdd': ['toImage'],  # Add the default download button
    'displaylogo': False  # Hide the Plotly logo in the mode bar
    }


    # Loop through the traces dictionary
    for trace_name, trace in traces.items():
        current_traces[trace_name] = trace  # Add trace to current_traces
        traces_count += 1

        
        # If the specified number of traces per figure is reached or all traces are processed, draw the figure
        if traces_count % num_traces_per_figure == 0 or traces_count == len(traces):
            fig = go.Figure(data=list(current_traces.values()))
            
            # Construct the figure title from the names of the traces
            figure_title = '__'.join(current_traces.keys())
            fig.update_layout(title=figure_title)  # Update figure title
            fig.update_layout(legend=dict(orientation='h')) # puts the legend at the bottom instead of to the right


            # Update filename in the config
            config['toImageButtonOptions']['filename'] = figure_title

            # Show the figure with the custom config
            pio.show(fig, config=config)

            current_traces = {}  # Clear current_traces for the next figure

    # Output the total number of figures created
    print(f'Total figures created: {figure_count - 1}')


In [ ]:
draw_traces(maps_t)

In [ ]:
draw_traces(rvsr_t)

In [ ]:
draw_traces(flip_t)

## variabel importance below

In [ ]:
def draw_predictor_correlations(filePath) 
    #
    # Look at correlation of predictors 
    #
    # pred = pd.read_csv("./data/surface_layer_model_predictions.csv"
    df = pd.read_csv(filePath)
    df['Time'] = pd.to_datetime(df['Time'])
    df.index = df['Time'] 

    # print(df.columns)
    
    # the subset below should be the inputs to the model, which should match what is in the yaml, 
    # so refactor to grab the appropriate yaml and draw correlation map of that - hector
    corr = df[[
                # 'temperature:12_m:K',
                'temperature:18.4_m:K', 
                'water_sfc_temperature:0_m:K'
                # # 'potential_temperature:12_m:K',
                'potential_temperature:18.4_m:K',
                'skin_virtual_potential_temperature:0_m:K', 
                'mixing_ratio:0_m:g_kg-1',
                # 'mixing_ratio:12_m:g_kg-1', 
                'mixing_ratio:18.4_m:g_kg-1'

                # 'relative_humidity:12_m:%',
                'relative_humidity:18.4_m:%',
                'wave_direction:0_m:degrees', 
                'wave_height:0_m:m', 
                'wave_period:0_m:s',
                'wave_phase_speed:0_m:m_s-1', 
                'near_surf_current:0_m:m_s-1',
                'near_surf_current_dir:0_m:deg', 
                'wind_speed:18.4_m:m_s-1',
                'wind_direction:18.4_m:degrees', 
                'angle_between_wind_wave:0_m:degrees',
                # 'pressure:12_m:hPa',
                'pressure:18.4_m:hPa',
                'u_wind:18.4_m:m_s-1', 
                'v_wind:18.4_m:m_s-1'
                # 'bulk_richardson:12_m:none'
                'bulk_richardson:18.4_m:none'
                ]].corr()
    #
    # Correlation map
    # 
    f, ax = plt.subplots(figsize=(15, 15))

    sns.heatmap(corr, mask=np.zeros_like(corr, dtype=np.bool),cmap=sns.diverging_palette(220, 10, as_cmap=True) ,
                square=True, ax=ax, annot=True, vmin= -1, vmax= 1)

    #sns.heatmap(corr[['next30Kt']].sort_values(by=['next30Kt'],ascending = False),cmap=sns.diverging_palette(220, 10, as_cmap=True) ,
    #          ax=ax, annot=True, vmin=-1, vmax = 1)
    cmap=sns.diverging_palette(220, 10, as_cmap=True)
    plt.show()
    plt.close()



#### this block of code only needs the input file to change, made it a function

In [ ]:
def draw_var_importance(filePath):
    df = pd.read_csv(filePath)
    df['aveImp'] = abs(df[[ 'all_0', 'all_1', 'all_2', 'all_3', 'all_4']].mean(axis = 1 ))

    print (df[['input', 'aveImp']].sort_values(by = 'aveImp'))

    barPlotImp= df[['input', 'aveImp']].sort_values(by = 'aveImp')
    ax = barPlotImp.plot.barh( color = 'blue',x ='input',y = 'aveImp', title =  'NN U* Pred Imp')
    

In [ ]:
draw_var_importance("models/model_QC_--kfold-0/heat_flux_neural_network_importances.csv")